# GHSL population surfaces (P2023A), clipped to the Russia–China border buffer

In [8]:
%pip install earthengine-api geemap pandas
import ee
import geemap
import pandas as pd


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
ee.Authenticate()
ee.Initialize(project='editorial-democraticprimaries')

In [10]:
# convert geojson to ee object
russiaChinaBorderObject = geemap.geojson_to_ee('china-russia-border.geojson')
russiaChinaBorderGeom = russiaChinaBorderObject.geometry()
russiaChinaBorderGeomBuffered = russiaChinaBorderGeom.buffer(50000)  # buffer by 100km

In [11]:
# GHSL population surfaces, 5-year epochs 1975-2030 (2025/2030 are projections), 100m
ghsPop = ee.ImageCollection('JRC/GHSL/P2023A/GHS_POP').select('population_count')
print(sorted(ghsPop.aggregate_array('system:index').getInfo()))

['1975', '1980', '1985', '1990', '1995', '2000', '2005', '2010', '2015', '2020', '2025', '2030']


In [12]:
buf = russiaChinaBorderGeomBuffered
ghsPopClipped = ghsPop.map(lambda img: img.clip(buf))
pop2020 = ghsPopClipped.filter(ee.Filter.eq('system:index', '2020')).first()

In [13]:
popVis = {'min': 0, 'max': 50, 'palette': ['000004', '51127c', 'b73779', 'fc8961', 'fcfdbf']}
m = geemap.Map(center=[48, 128], zoom=5)
m.add_tile_layer('https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}&hl=en', name='Google Roads', attribution='Google')
m.addLayer(pop2020, popVis, 'GHS-POP 2020 (people per 100m cell)')
m.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(buf)]), 1, 2), {'palette': 'white'}, 'Border buffer (100km)')
# GHSL epochs are 5-yearly, so 2015 stands in for 2016 (2025 is a projection)
pop2015 = ghsPopClipped.filter(ee.Filter.eq('system:index', '2015')).first()
pop2025 = ghsPopClipped.filter(ee.Filter.eq('system:index', '2025')).first()
DENOM_FLOOR = 1  # people per cell; avoids huge % where baseline is ~0
popPctChange = pop2025.subtract(pop2015).divide(pop2015.max(DENOM_FLOOR)).multiply(100)
changeVis = {'min': -100, 'max': 100, 'palette': ['b2182b', 'f7f7f7', '2166ac']}
m.addLayer(popPctChange, changeVis, 'Pop % change 2015-2025')
m.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(russiaChinaBorderGeom)]), 1, 2), {'palette': 'black'}, 'Border')
m

Map(center=[48, 128], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

In [14]:
# sanity check: total population inside the buffer per epoch
def total(img):
    s = img.reduceRegion(ee.Reducer.sum(), buf, 100, maxPixels=1e13)
    return ee.Feature(None, {'year': img.get('system:index'), 'population': s.get('population_count')})

rows = ghsPop.map(total).getInfo()['features']
popTotals = pd.DataFrame([r['properties'] for r in rows]).sort_values('year').reset_index(drop=True)
popTotals

,population,year
0,-4.951713e+07,1975
1,-4.912548e+07,1980
2,-4.868376e+07,1985
3,-4.818858e+07,1990
4,-4.794851e+07,1995
5,-4.781246e+07,2000
6,-4.788901e+07,2005
7,-4.796925e+07,2010
8,-4.791015e+07,2015
9,-4.783741e+07,2020
